# 20 — Face Anti-Spoofing Streamlit 앱 실행
> **모델:** `stage2_webcam_v4.h5` (threshold=0.5463)  
> **XAI:** Grad-CAM + 수치 앵커링 + LLaVA 캡션 (3계층)  
> **실행:** ngrok으로 외부 URL 생성 → 브라우저에서 접속

---
## 📋 체크리스트
- [ ] Cell 1: Drive 마운트 + 패키지 설치
- [ ] Cell 2: streamlit_app.py 생성 (Drive에 저장)
- [ ] Cell 3: ngrok 토큰 설정 & 앱 실행
- [ ] Cell 4: (선택) 웹캠 이미지로 빠른 동작 테스트

## Cell 1 — Drive 마운트 + 패키지 설치

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE       = '/content/drive/MyDrive/face-anti-spoofing'
MODEL_DIR  = f'{BASE}/models'
SRC_DIR    = f'{BASE}/src'
APP_DIR    = f'{BASE}/app'
CAPTION_JSON = f'{BASE}/results/phase4/llava_captions.json'

os.makedirs(APP_DIR, exist_ok=True)
os.makedirs(SRC_DIR, exist_ok=True)

print('패키지 설치 중...')
os.system('pip install streamlit pyngrok -q')
print('✅ 완료')

import tensorflow as tf
print('TF:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## Cell 2 — streamlit_app.py 생성

> Drive의 `app/streamlit_app.py`로 저장  
> Colab 런타임에 심링크로 연결해서 실행

In [ ]:
APP_CODE = '''
import streamlit as st
import numpy as np
import cv2
import tensorflow as tf
import json
import random
from pathlib import Path

# ── 설정 ──────────────────────────────────────────────────────
BASE         = "/content/drive/MyDrive/face-anti-spoofing"
MODEL_PATH   = f"{BASE}/models/stage2_webcam_v4.h5"
CAPTION_JSON = f"{BASE}/results/phase4/llava_captions.json"
THRESHOLD    = 0.5463

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

SPOOF_KO = {
    0: "Live (실제 얼굴)",
    1: "Print Attack (인쇄 공격)",
    2: "Replay Attack (화면 재촬영)",
    3: "3D Mask (입체 마스크)",
}
SPOOF_EN_HINT = {
    1: "인쇄물 특유의 평탄한 피부 질감과 낮은 고주파 에너지가 관측됩니다.",
    2: "화면 재촬영 특유의 모아레 패턴 및 고주파 에너지 감소가 감지됩니다.",
    3: "마스크 경계부에 비정상적 선명도 및 피부 질감 불일치가 나타납니다.",
}
ANCHOR_BASELINE = {
    "live"  : {"laplacian": 383, "fft_high": 1134},
    "print" : {"laplacian": 318, "fft_high": 1042},
    "replay": {"laplacian": 319, "fft_high":  944},
    "mask"  : {"laplacian": 480, "fft_high": 1134},
}

# ── 모델 로드 (캐시) ──────────────────────────────────────────
@st.cache_resource
def load_model():
    return tf.keras.models.load_model(MODEL_PATH)

# ── 캡션 DB 로드 (캐시) ───────────────────────────────────────
@st.cache_resource
def load_captions():
    db, pool = {}, {"live": [], "print": [], "replay": [], "mask": []}
    if Path(CAPTION_JSON).exists():
        with open(CAPTION_JSON) as f:
            records = json.load(f)
        for rec in records.get("results", []):
            stem = Path(rec["img_path"]).stem
            cap  = rec.get("caption", "")
            db[stem] = cap
            cat = rec.get("category", "")
            if cat in pool and cap:
                pool[cat].append(cap)
    return db, pool

# ── 전처리 ────────────────────────────────────────────────────
def preprocess(img_bgr):
    img = cv2.resize(img_bgr, (224, 224))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    img = (img - IMAGENET_MEAN) / IMAGENET_STD
    return np.expand_dims(img, 0)

# ── Grad-CAM ──────────────────────────────────────────────────
def get_gradcam(model, inp):
    conv_layer  = None
    logit_layer = "binary"
    for layer in reversed(model.layers):
        if isinstance(layer, tf.keras.layers.Conv2D):
            conv_layer = layer.name
            break
    if conv_layer is None:
        return np.zeros((224, 224))

    grad_model = tf.keras.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(conv_layer).output,
                 model.get_layer(logit_layer).output]
    )
    with tf.GradientTape() as tape:
        x = tf.cast(inp, tf.float32)
        conv_out, pred = grad_model(x)
        p = tf.clip_by_value(pred[:, 0], 1e-7, 1 - 1e-7)
        loss = tf.math.log(p / (1.0 - p))
    grads   = tape.gradient(loss, conv_out)
    pooled  = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = conv_out[0] @ pooled[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0)
    heatmap = heatmap / (tf.math.reduce_max(heatmap) + 1e-8)
    return cv2.resize(heatmap.numpy(), (224, 224))

def overlay_heatmap(img_bgr, heatmap, alpha=0.45):
    h, w  = img_bgr.shape[:2]
    hmr   = cv2.resize(heatmap, (w, h))
    hmc   = cv2.applyColorMap(np.uint8(255 * hmr), cv2.COLORMAP_JET)
    img_r = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    hmc_r = cv2.cvtColor(hmc,     cv2.COLOR_BGR2RGB)
    return cv2.addWeighted(img_r, 1 - alpha, hmc_r, alpha, 0)

# ── 수치 앵커링 ───────────────────────────────────────────────
def compute_pixel_features(img_bgr, mask=None):
    img_224 = cv2.resize(img_bgr, (224, 224))
    gray    = cv2.cvtColor(img_224, cv2.COLOR_BGR2GRAY).astype(np.float32)
    if mask is not None and mask.any():
        gray_m = gray.copy()
        gray_m[~mask] = 0
    else:
        gray_m = gray
    lap = cv2.Laplacian(gray_m.astype(np.uint8), cv2.CV_64F).var()
    f   = np.fft.fft2(gray_m)
    mag = np.abs(np.fft.fftshift(f))
    h, w = mag.shape
    cy, cx = h // 2, w // 2
    r = min(h, w) // 4
    mk = np.ones((h, w), np.uint8)
    cv2.circle(mk, (cx, cy), r, 0, -1)
    fft_high = (mag * mk).sum() / (mag.sum() + 1e-8) * 1000
    return {"laplacian": round(float(lap), 1), "fft_high": round(float(fft_high), 1)}

def interpret_anchors(stats, cat_key):
    base  = ANCHOR_BASELINE.get(cat_key, ANCHOR_BASELINE["live"])
    lines = []
    ld = stats["laplacian"] - base["laplacian"]
    if   ld < -40: lines.append("질감 평탄 (선명도 낮음)")
    elif ld >  40: lines.append("경계선 강조 (선명도 높음)")
    else:          lines.append("선명도 보통")
    fd = stats["fft_high"] - base["fft_high"]
    if   fd < -80: lines.append("고주파 에너지 부재 (압축/재촬영 흔적)")
    elif fd >  80: lines.append("고주파 에너지 과잉 (경계 아티팩트)")
    else:          lines.append("고주파 에너지 정상")
    return " / ".join(lines)

# ── explain() ─────────────────────────────────────────────────
def explain(img_bgr, model, caption_db, caption_pool):
    inp   = preprocess(img_bgr)
    preds = model.predict(inp, verbose=0)

    if isinstance(preds, list) and len(preds) >= 2:
        spoof_prob     = float(preds[0][0][0])
        spoof_type_idx = int(np.argmax(preds[1][0]))
    else:
        spoof_prob     = float(preds[0][0]) if preds.ndim > 1 else float(preds[0])
        spoof_type_idx = 0

    verdict         = "FAKE" if spoof_prob >= THRESHOLD else "REAL"
    spoof_type_name = SPOOF_KO.get(spoof_type_idx, f"유형 {spoof_type_idx}")
    cat_key         = {1: "print", 2: "replay", 3: "mask"}.get(spoof_type_idx, "live")

    heatmap_raw     = get_gradcam(model, inp)
    heatmap_overlay = overlay_heatmap(img_bgr, heatmap_raw)

    mask_224    = cv2.resize(heatmap_raw, (224, 224)) >= 0.4
    anchor_stats  = compute_pixel_features(img_bgr, mask_224 if mask_224.any() else None)
    anchor_interp = interpret_anchors(anchor_stats, cat_key)

    # 캡션: 풀에서 랜덤
    pool    = caption_pool.get(cat_key, [])
    caption = random.choice(pool) if pool else ""

    # XAI 텍스트
    verdict_ko = "🟢 실제 얼굴 (REAL)" if verdict == "REAL" else "🔴 위조 공격 감지 (FAKE)"
    xai_lines  = [
        f"**{verdict_ko}** — 신뢰도 {spoof_prob:.1%}",
        f"**예측 공격 유형:** {spoof_type_name}",
        "",
        "**[Layer 1 — Grad-CAM]** 어디를 봤는가",
        "> 히트맵: 얼굴 중심부 고활성 영역 (logit 기반)",
        "",
        "**[Layer 2 — 수치 앵커링]** 얼마나 강한 신호인가",
        f"> Laplacian: **{anchor_stats[\"laplacian\"]}**  |  FFT 고주파: **{anchor_stats[\"fft_high\"]}**",
        f"> 해석: {anchor_interp}",
    ]
    hint = SPOOF_EN_HINT.get(spoof_type_idx)
    if hint and verdict == "FAKE":
        xai_lines.append(f"> ✏️ {hint}")
    xai_lines += [
        "",
        "**[Layer 3 — VLM 자연어 분석]** 왜 그렇게 판단했는가",
        f"> {caption}" if caption else "> *(캡션 없음)*",
    ]

    return {
        "verdict"        : verdict,
        "spoof_prob"     : spoof_prob,
        "spoof_type_name": spoof_type_name,
        "heatmap_overlay": heatmap_overlay,
        "anchor_stats"   : anchor_stats,
        "xai_text"       : "\\n".join(xai_lines),
    }

# ── Streamlit UI ──────────────────────────────────────────────
st.set_page_config(
    page_title="Face Anti-Spoofing",
    page_icon="🛡️",
    layout="wide"
)

st.title("🛡️ Face Anti-Spoofing — XAI 데모")
st.caption("AI Security & Application · 단국대학교 · stage2_webcam_v4 (threshold=0.5463)")
st.divider()

model       = load_model()
caption_db, caption_pool = load_captions()

uploaded = st.file_uploader(
    "이미지 업로드 (jpg / png)",
    type=["jpg", "jpeg", "png"]
)

if uploaded:
    file_bytes = np.frombuffer(uploaded.read(), np.uint8)
    img_bgr    = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)

    with st.spinner("분석 중..."):
        result = explain(img_bgr, model, caption_db, caption_pool)

    # ── 결과 레이아웃 (3열) ───────────────────────────────────
    col1, col2, col3 = st.columns(3)

    with col1:
        st.subheader("원본 이미지")
        st.image(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB), use_column_width=True)

    with col2:
        st.subheader("Grad-CAM 히트맵")
        st.image(result["heatmap_overlay"], use_column_width=True)
        verdict = result["verdict"]
        prob    = result["spoof_prob"]
        if verdict == "REAL":
            st.success(f"✅ REAL — {prob:.1%}")
        else:
            st.error(f"🚨 FAKE — {prob:.1%}")
        st.caption(f"예측 유형: {result[\"spoof_type_name\"]}")

    with col3:
        st.subheader("XAI 설명")
        st.markdown(result["xai_text"])

    # ── 수치 상세 ─────────────────────────────────────────────
    with st.expander("📊 수치 상세 보기"):
        a = result["anchor_stats"]
        c1, c2 = st.columns(2)
        c1.metric("Laplacian 분산", a["laplacian"])
        c2.metric("FFT 고주파 에너지", a["fft_high"])

else:
    st.info("👆 이미지를 업로드하면 Real/Fake 판정 및 XAI 설명이 표시됩니다.")
    st.markdown("""
    **테스트 방법:**
    - 웹캠으로 찍은 본인 얼굴 사진 → REAL
    - 모니터에 띄운 얼굴 사진 촬영 → Replay Attack
    - 프린트된 얼굴 사진 촬영 → Print Attack
    """)
'''

# Drive에 저장
app_path = f'{APP_DIR}/streamlit_app.py'
with open(app_path, 'w', encoding='utf-8') as f:
    f.write(APP_CODE)

# Colab 로컬에도 복사 (실행용)
os.system(f'cp "{app_path}" /content/streamlit_app.py')

print('✅ streamlit_app.py 생성 완료')
print(f'   Drive: {app_path}')
print(f'   Local: /content/streamlit_app.py')

## Cell 3 — ngrok 설정 & Streamlit 실행

**ngrok 토큰:** https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
import subprocess, time
from pyngrok import ngrok, conf

# ── ⚠️ 본인 ngrok 토큰으로 교체 ──────────────────────────────
NGROK_TOKEN = ''   # ← 여기에 입력

assert NGROK_TOKEN, '❌ NGROK_TOKEN을 입력하세요 (https://dashboard.ngrok.com)'

conf.get_default().auth_token = NGROK_TOKEN

# 기존 프로세스 정리
os.system('pkill -f streamlit 2>/dev/null; sleep 1')
ngrok.kill()

# Streamlit 백그라운드 실행
proc = subprocess.Popen(
    ['streamlit', 'run', '/content/streamlit_app.py',
     '--server.port', '8501',
     '--server.headless', 'true',
     '--server.enableCORS', 'false'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(4)

# ngrok 터널 생성
tunnel = ngrok.connect(8501)
print('=' * 50)
print(f'🌐 접속 URL: {tunnel.public_url}')
print('=' * 50)
print('위 URL을 브라우저에서 열어보세요!')
print('종료하려면: ngrok.kill() 실행')

## Cell 4 — (선택) 빠른 동작 테스트

브라우저 없이 코드 레벨에서 `explain()` 동작 확인

In [ ]:
import cv2
import numpy as np
import tensorflow as tf
from pathlib import Path
import matplotlib.pyplot as plt

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
THRESHOLD     = 0.5463
SPOOF_KO = {0:'Live',1:'Print Attack',2:'Replay Attack',3:'3D Mask'}

model = tf.keras.models.load_model(f'{MODEL_DIR}/stage2_webcam_v4.h5')
print('✅ 모델 로드 완료')

# webcam 이미지로 테스트
test_sets = [
    (f'{BASE}/data/webcam_live',   'REAL'),
    (f'{BASE}/data/webcam_print',  'FAKE'),
    (f'{BASE}/data/webcam_replay', 'FAKE'),
]

fig, axes = plt.subplots(3, 2, figsize=(10, 12))

for row, (folder, expected) in enumerate(test_sets):
    paths = sorted(Path(folder).glob('*.jpg'))[:1] + \
            sorted(Path(folder).glob('*.png'))[:1]
    if not paths:
        print(f'⚠️  {folder} 이미지 없음')
        continue

    img_bgr = cv2.imread(str(paths[0]))
    img_224 = cv2.resize(img_bgr, (224, 224))

    # 전처리 & 추론
    inp   = np.expand_dims(
        (cv2.cvtColor(img_224, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
         - IMAGENET_MEAN) / IMAGENET_STD, 0
    )
    preds = model.predict(inp, verbose=0)
    if isinstance(preds, list):
        prob = float(preds[0][0][0])
        stype = int(np.argmax(preds[1][0]))
    else:
        prob  = float(preds[0][0])
        stype = 0

    verdict = 'FAKE' if prob >= THRESHOLD else 'REAL'
    match   = '✅' if verdict == expected else '❌'

    # 원본
    axes[row][0].imshow(cv2.cvtColor(img_224, cv2.COLOR_BGR2RGB))
    axes[row][0].set_title(
        f'{Path(folder).name}\n{match} {verdict} ({prob:.1%}) | 예상: {expected}',
        fontsize=9,
        color='green' if verdict == expected else 'red'
    )
    axes[row][0].axis('off')

    # Grad-CAM
    try:
        conv_layer = next(
            l.name for l in reversed(model.layers)
            if isinstance(l, tf.keras.layers.Conv2D)
        )
        gm = tf.keras.Model(
            inputs=model.inputs,
            outputs=[model.get_layer(conv_layer).output,
                     model.get_layer('binary').output]
        )
        with tf.GradientTape() as tape:
            x = tf.cast(inp, tf.float32)
            co, pr = gm(x)
            p = tf.clip_by_value(pr[:, 0], 1e-7, 1 - 1e-7)
            loss = tf.math.log(p / (1.0 - p))
        grads = tape.gradient(loss, co)
        pooled = tf.reduce_mean(grads, axis=(0, 1, 2))
        hm = tf.maximum(co[0] @ pooled[..., tf.newaxis], 0)
        hm = tf.squeeze(hm)
        hm = (hm / (tf.math.reduce_max(hm) + 1e-8)).numpy()
        hm_r = cv2.resize(hm, (224, 224))
        jet  = cv2.applyColorMap(np.uint8(255 * hm_r), cv2.COLORMAP_JET)
        over = cv2.addWeighted(
            cv2.cvtColor(img_224, cv2.COLOR_BGR2RGB), 0.55,
            cv2.cvtColor(jet,     cv2.COLOR_BGR2RGB), 0.45, 0
        )
        axes[row][1].imshow(over)
        axes[row][1].set_title(f'Grad-CAM | {SPOOF_KO[stype]}', fontsize=9)
    except Exception as e:
        axes[row][1].set_title(f'오류: {e}', fontsize=7)
    axes[row][1].axis('off')

plt.suptitle('Cell 4 — 빠른 동작 테스트', fontsize=13)
plt.tight_layout()
plt.savefig(f'{BASE}/reports/phase5/20_quick_test.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ 테스트 완료')